In [1]:
# Task 1 — Data Loading & Exploration

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv(r'D:\projects\HousePricePrediction_Vishnu Thota\data\raw\Housing.csv')

print("\nFirst 10 rows:")
display(df.head(10))

print('\nRows and Columns in the dataset:', df.shape)

print("\nColumn names and data types:\n")
print(df.dtypes)
print("Target column : price")
print("Feature column : ", [col for col in df.columns if col != 'price'])

print("\nMissing values per column:")
print(df.isnull().sum())

print(f"\nTotal missing values: {df.isnull().sum().sum()}")


First 10 rows:


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished
5,10850000,7500,3,3,1,yes,no,yes,no,yes,2,yes,semi-furnished
6,10150000,8580,4,3,4,yes,no,no,no,yes,2,yes,semi-furnished
7,10150000,16200,5,3,2,yes,no,no,no,no,0,no,unfurnished
8,9870000,8100,4,1,2,yes,yes,yes,no,yes,2,yes,furnished
9,9800000,5750,3,2,4,yes,yes,no,no,yes,1,yes,unfurnished



Rows and Columns in the dataset: (545, 13)

Column names and data types:

price                int64
area                 int64
bedrooms             int64
bathrooms            int64
stories              int64
mainroad            object
guestroom           object
basement            object
hotwaterheating     object
airconditioning     object
parking              int64
prefarea            object
furnishingstatus    object
dtype: object
Target column : price
Feature column :  ['area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus']

Missing values per column:
price               0
area                0
bedrooms            0
bathrooms           0
stories             0
mainroad            0
guestroom           0
basement            0
hotwaterheating     0
airconditioning     0
parking             0
prefarea            0
furnishingstatus    0
dtype: int64

Total missing values: 0


In [2]:
# Task 2 — Data Cleaning

print(f'Duplicate rows: {df.duplicated().sum()}')

category_cols = ['mainroad', 'guestroom', 'basement', 
               'hotwaterheating', 'airconditioning', 'prefarea']
df[category_cols] = df[category_cols].apply(lambda x: x.map({'yes':1, 'no':0}))

print(df[category_cols].head())

df = pd.get_dummies(df, columns = ['furnishingstatus'], drop_first = True)

bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print("\nColumns after encoding:")
print(df.columns.tolist())

print("Data types after cleaning:")
print(df.dtypes)

print(f"\nAny remaining non-numeric columns: "
      f"{df.select_dtypes(include='object').columns.tolist()}")

print(f"\nFinal shape: {df.shape}")
print(f"\nMissing values: {df.isnull().sum().sum()}")

Duplicate rows: 0
   mainroad  guestroom  basement  hotwaterheating  airconditioning  prefarea
0         1          0         0                0                1         1
1         1          0         0                0                1         0
2         1          0         1                0                0         1
3         1          0         1                0                1         1
4         1          1         1                0                1         0

Columns after encoding:
['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus_semi-furnished', 'furnishingstatus_unfurnished']
Data types after cleaning:
price                              int64
area                               int64
bedrooms                           int64
bathrooms                          int64
stories                            int64
mainroad                           int64
gue

In [3]:
df['total_rooms'] = df['bedrooms'] + df['bathrooms']
df['area_per_room'] = df['area'] / (df['bedrooms'] + df['bathrooms'])
df['luxury_score'] = (df['airconditioning'] + 
                      df['hotwaterheating'] + 
                      df['prefarea']        + 
                      df['guestroom']       +
                      df['basement'])

print("New features created:")
print(df[['total_rooms', 'area_per_room', 'luxury_score']].describe())
print(f"\nDataframe shape after feature engineering: {df.shape}")
print(f"\nAll columns now:\n{df.columns.tolist()}")

New features created:
       total_rooms  area_per_room  luxury_score
count   545.000000     545.000000    545.000000
mean      4.251376    1257.549244      1.124771
std       1.036611     567.703083      1.084242
min       2.000000     317.500000      0.000000
25%       4.000000     883.333333      0.000000
50%       4.000000    1160.000000      1.000000
75%       5.000000    1500.000000      2.000000
max       8.000000    4400.000000      4.000000

Dataframe shape after feature engineering: (545, 17)

All columns now:
['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus_semi-furnished', 'furnishingstatus_unfurnished', 'total_rooms', 'area_per_room', 'luxury_score']


In [4]:
# Task 3 — Model Building

X = df.drop('price', axis = 1)
y = df['price']

print("Features shape:", X.shape)
print("Target shape:", y.shape)
print("\nFeatures being used:")
print(X.columns.tolist())

Features shape: (545, 16)
Target shape: (545,)

Features being used:
['area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning', 'parking', 'prefarea', 'furnishingstatus_semi-furnished', 'furnishingstatus_unfurnished', 'total_rooms', 'area_per_room', 'luxury_score']


In [5]:
# Train/Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
print(f"\nTraining set  : {X_train.shape[0]} rows")
print(f"Test set      : {X_test.shape[0]} rows")


Training set  : 436 rows
Test set      : 109 rows


In [6]:
#Feature Scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\nScaling complete.")
print(f"Training mean: {X_train_scaled.mean():.4f}")
print(f"Training std: {X_train_scaled.std():.4f}")



Scaling complete.
Training mean: -0.0000
Training std: 1.0000


In [7]:
# Linear Regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
lr_predictions = lr_model.predict(X_test_scaled)

lr_mae = mean_absolute_error(y_test, lr_predictions)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_predictions))
lr_r2 = r2_score(y_test, lr_predictions)

print("=== Linear Regression Results ===")
print(f"MAE  : ₹{lr_mae:,.0f}")
print(f"RMSE : ₹{lr_rmse:,.0f}")
print(f"R²   : {lr_r2:.4f}")

=== Linear Regression Results ===
MAE  : ₹966,320
RMSE : ₹1,314,226
R²   : 0.6583


In [8]:
# Random Forest
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators' : [100, 200, 300],
    'max_depth'    : [3, 5, 7, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4]
}

rf_model = RandomForestRegressor(random_state = 42)
grid_search = GridSearchCV(estimator = rf_model, param_grid = param_grid, cv = 5, n_jobs = -1, verbose = 1)

grid_search.fit(X_train, y_train)

print("Best parameters found:")
print(grid_search.best_params_)
print(f"\nBest cross-validated R²: {grid_search.best_score_:.4f}")

Fitting 5 folds for each of 108 candidates, totalling 540 fits
Best parameters found:
{'max_depth': 7, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 100}

Best cross-validated R²: 0.6351


In [10]:
# Training final model with best parameters
rf_model = grid_search.best_estimator_

y_pred_rf = rf_model.predict(X_test)

mae_rf  = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf   = r2_score(y_test, y_pred_rf)

print("Tuned Random Forest Results")
print(f"MAE  : ₹{mae_rf:,.0f}")
print(f"RMSE : ₹{rmse_rf:,.0f}")
print(f"R²   : {r2_rf:.4f}")

Tuned Random Forest Results
MAE  : ₹1,049,055
RMSE : ₹1,426,042
R²   : 0.5977


In [11]:
display(df.head())

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus_semi-furnished,furnishingstatus_unfurnished,total_rooms,area_per_room,luxury_score
0,13300000,7420,4,2,3,1,0,0,0,1,2,1,0,0,6,1236.666667,2
1,12250000,8960,4,4,4,1,0,0,0,1,3,0,0,0,8,1120.000000,1
2,12250000,9960,3,2,2,1,0,1,0,0,2,1,1,0,5,1992.000000,2
3,12215000,7500,4,2,2,1,0,1,0,1,3,1,0,0,6,1250.000000,3
4,11410000,7420,4,1,2,1,1,1,0,1,2,0,0,0,5,1484.000000,3


In [14]:
import joblib

joblib.dump(lr_model, '../models/linear_regression_model.pkl')
joblib.dump(scaler,   '../models/scaler.pkl')

# Save tuned Random Forest
joblib.dump(rf_model, '../models/random_forest_model.pkl')

print("Models saved successfully:")
print("  models/linear_regression_model.pkl")
print("  models/scaler.pkl")
print("  models/random_forest_model.pkl")

Models saved successfully:
  models/linear_regression_model.pkl
  models/scaler.pkl
  models/random_forest_model.pkl
